# Notebook 04: QLoRA (Quantized Low-Rank Adaptation)

### Research Questions:
> **RQ3:** *How does QLoRA compare with LoRA in predictive performance?*
> **RQ4:** *How much GPU memory does QLoRA save compared with LoRA?*

### What is QLoRA?

QLoRA quantizes the frozen base model weights $W$ into **4-bit NormalFloat (NF4)** precision:

$$W_q = Q(W)$$

The adapted forward pass computes:

$$\boxed{W_{\text{effective}} \approx W_q + \frac{\alpha}{r} B A}$$

### Understanding Quantization Error

Quantizing from 16-bit to 4-bit causes a quantization discrepancy:

$$E_q = W - W_q$$

QLoRA does **not** modify the frozen quantized weights to directly eliminate $E_q$.
Instead, backpropagation updates only the low-rank adapter matrices $A$ and $B$ to minimize task loss on the adapted model:

$$\boxed{\min_{A, B} \mathcal{L}\left(W_q + \frac{\alpha}{r} B A\right)}$$

### Fair Comparison Protocol
To ensure scientific rigor, all hyperparameters (rank $r=16$, $\alpha=32$, dropout, target modules, learning rate, batch size, epochs, dataset split) are **strictly identical** to Notebook 03.

In [1]:
# === MOUNT GOOGLE DRIVE & SETUP ===
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment')
sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'peft', 'accelerate', 'bitsandbytes',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'pyyaml'],
    check=True)

for d in ['results/metrics', 'results/predictions', 'results/figures',
          'results/logs', 'adapters/lora', 'adapters/qlora']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'src/ found: {os.path.exists("src")}')  # Must be True!
print('Setup complete! ✅')

Mounted at /content/drive
Working directory: /content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment
src/ found: True
Setup complete! ✅


In [2]:
# Step 2: Hardware check
import json
import torch


from src.model_utils import detect_hardware, print_hardware_summary, check_vram_feasibility

hw = detect_hardware()
print_hardware_summary(hw)
# Primary research model: Qwen2.5-3B-Instruct
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
is_feasible, warning_msg = check_vram_feasibility(MODEL_NAME, method='qlora', hardware_info=hw)
print(warning_msg)
print(f'Active model: {MODEL_NAME} for 4-bit NF4 QLoRA fine-tuning.')

           HARDWARE & RUNTIME ENVIRONMENT SUMMARY           
CUDA Available:        True
GPU Name:              Tesla T4
GPU VRAM:              14.56 GB (14912.69 MB)
CUDA Version:          12.8
BF16 Supported:        True
PyTorch Version:       2.11.0+cu128
Transformers Version:  5.16.1
Compute Dtype:         bfloat16
Hardware check PASSED: 14.56 GB VRAM is sufficient for Qwen/Qwen2.5-3B-Instruct (QLORA).
Active model: Qwen/Qwen2.5-3B-Instruct for 4-bit NF4 QLoRA fine-tuning.


In [3]:
# Step 3: Load dataset
from src.data_utils import load_raw_dataset, create_stratified_split, prepare_hf_training_dataset
from src.model_utils import load_tokenizer

tokenizer = load_tokenizer(MODEL_NAME)
df = load_raw_dataset('takala/financial_phrasebank', 'sentences_50agree')
train_df, val_df, test_df = create_stratified_split(df, 0.70, 0.15, 0.15, random_seed=42)

train_dataset = prepare_hf_training_dataset(train_df, tokenizer=tokenizer, max_length=256)
val_dataset = prepare_hf_training_dataset(val_df, tokenizer=tokenizer, max_length=256)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.58k [00:00<?, ?B/s]

financial_phrasebank.py:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

Loading via `datasets` failed (Dataset scripts are no longer supported, but found financial_phrasebank.py). Using direct Hugging Face Hub download fallback...


In [4]:
# Step 4: Load 4-Bit Quantized Base Model and Apply LoRA
from src.model_utils import load_base_model, apply_lora_to_model, count_parameters
from src.memory_utils import MemoryTracker, get_gpu_memory_stats, reset_memory_stats

reset_memory_stats()
print(f'Loading {MODEL_NAME} with 4-bit NormalFloat (NF4) + Double Quantization...')

with MemoryTracker('4-Bit Model Load') as q_load_tracker:
    q_base_model = load_base_model(MODEL_NAME, quantization=True)

print(f'Loaded base model into 4-bit! {q_load_tracker.summary()}')

print('Attaching LoRA adapter (identical configuration: r=16, alpha=32)...')
q_model = apply_lora_to_model(
    q_base_model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    is_qlora=True,
    gradient_checkpointing=True,
)

param_stats = count_parameters(q_model)
print('=== Parameter Footprint ===')
print(f'Total Parameters:     {param_stats["total_parameters_millions"]} M')
print(f'Trainable Parameters: {param_stats["trainable_parameters_millions"]} M ({param_stats["trainable_percentage"]} %)')

Loading Qwen/Qwen2.5-3B-Instruct with 4-bit NormalFloat (NF4) + Double Quantization...


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded base model into 4-bit! [4-Bit Model Load] Peak Allocated: 2009.91 MB (1.963 GB) | Peak Reserved: 2028.0 MB (1.98 GB)
Attaching LoRA adapter (identical configuration: r=16, alpha=32)...
=== Parameter Footprint ===
Total Parameters:     1706.05 M
Trainable Parameters: 7.3728 M (0.4322 %)


In [5]:
# Tokenize the datasets (fixes the ['label'] error)
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

if "text" in train_dataset.column_names:
    train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text", "label"])
    val_dataset = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text", "label"])
    print(f"Tokenized! Columns: {train_dataset.column_names}")
else:
    print(f"Already tokenized! Columns: {train_dataset.column_names}")

Map:   0%|          | 0/3392 [00:00<?, ? examples/s]

Map:   0%|          | 0/727 [00:00<?, ? examples/s]

Tokenized! Columns: ['input_ids', 'attention_mask']


In [6]:
# Step 5: Train QLoRA Model
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from transformers.trainer_utils import get_last_checkpoint
import os
import time

# ============================================================
# COLAB-SAFE CHECKPOINTING
# Checkpoints are stored permanently on Google Drive
# ============================================================

CHECKPOINT_DIR = (
    '/content/drive/MyDrive/llm-project/'
    'lora-qlora-financial-sentiment/checkpoints/qlora'
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  # Effective batch size = 8
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy='epoch',

    # Save checkpoint every 100 training steps
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,

    bf16=hw['bf16_supported'],
    fp16=not hw['bf16_supported'] and hw['cuda_available'],
    gradient_checkpointing=True,
    report_to='none',
)

trainer = Trainer(
    model=q_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer,
        mlm=False
    ),
)

# ============================================================
# AUTOMATIC CHECKPOINT RESUME
# ============================================================

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if last_checkpoint is not None:
    print(f'Latest checkpoint found: {last_checkpoint}')
    print('Resuming QLoRA training from the latest checkpoint...')
else:
    print('No checkpoint found.')
    print('Starting QLoRA training from scratch.')

# ============================================================
# START / RESUME TRAINING
# ============================================================

print('Starting QLoRA training with memory tracking...')

with MemoryTracker('QLoRA Training') as q_train_tracker:
    t0 = time.perf_counter()

    trainer.train(
        resume_from_checkpoint=last_checkpoint
    )

    training_duration = time.perf_counter() - t0

print(f'QLoRA Training finished in {training_duration/60:.2f} minutes.')
print(q_train_tracker.summary())

Latest checkpoint found: /content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment/checkpoints/qlora/checkpoint-1272
Resuming QLoRA training from the latest checkpoint...
Starting QLoRA training with memory tracking...


Epoch,Training Loss,Validation Loss


QLoRA Training finished in 0.11 minutes.
[QLoRA Training] Peak Allocated: 2648.81 MB (2.587 GB) | Peak Reserved: 3300.0 MB (3.223 GB)


In [7]:
# Step 6: Save QLoRA Adapter
from src.model_utils import get_adapter_disk_size_mb

adapter_dir = 'adapters/qlora'
os.makedirs(adapter_dir, exist_ok=True)
q_model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

adapter_size = get_adapter_disk_size_mb(adapter_dir)
print(f'QLoRA adapter saved to {adapter_dir} ({adapter_size} MB)')

QLoRA adapter saved to adapters/qlora (39.06 MB)


In [8]:
# Step 7: Evaluate QLoRA on Fixed Test Set
from src.data_utils import format_chat_prompt
from src.metrics import parse_model_output, compute_classification_metrics, format_metrics_summary
from src.visualization import plot_confusion_matrix_heatmap
import numpy as np

device = next(q_model.parameters()).device
raw_outputs = []
parsed_preds = []
latencies = []

q_model.eval()
print(f'Evaluating fine-tuned QLoRA model on {len(test_df)} test samples...')

for idx, row in test_df.iterrows():
    prompt = format_chat_prompt(row['sentence'], tokenizer=tokenizer)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    t0 = time.perf_counter()
    with torch.no_grad():
        out = q_model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000.0)

    gen_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    raw_outputs.append(gen_text)
    parsed_preds.append(parse_model_output(gen_text))

metrics = compute_classification_metrics(test_df['label_text'].tolist(), parsed_preds)
metrics['method'] = 'qlora'
metrics['model_name'] = MODEL_NAME
metrics['trainable_parameters'] = param_stats['trainable_parameters']
metrics['trainable_percentage'] = param_stats['trainable_percentage']
metrics['adapter_size_mb'] = adapter_size
metrics['training_time_seconds'] = round(training_duration, 2)
metrics['peak_vram_gb'] = q_train_tracker.peak_allocated_gb
metrics['latency_median_ms'] = round(float(np.median(latencies)), 2)

print(format_metrics_summary(metrics, 'QLoRA Fine-Tuned Model'))

with open('results/metrics/qlora.json', 'w') as f:
    json.dump(metrics, f, indent=2)

test_df_copy = test_df.copy()
test_df_copy['predicted_label'] = parsed_preds
test_df_copy['raw_output'] = raw_outputs
test_df_copy.to_csv('results/predictions/qlora.csv', index=False)

plot_confusion_matrix_heatmap(metrics['confusion_matrix'], method_name='QLoRA Fine-Tuned')
print('Saved results to results/metrics/qlora.json and results/predictions/qlora.csv')

Evaluating fine-tuned QLoRA model on 727 test samples...
        CLASSIFICATION PERFORMANCE: QLORA FINE-TUNED MODEL
Accuracy:            0.8569 (85.69%)
Macro F1:            0.8475
Macro Precision:     0.8418
Macro Recall:        0.8596
Weighted F1:         0.8554
Invalid Output Rate: 0.00% (0/727)
-------------------------------------------------------
Per-Class Performance:
  Negative  ->  P: 0.8037 | R: 0.9451 | F1: 0.8687 (n=91)
  Neutral   ->  P: 0.8733 | R: 0.8935 | F1: 0.8833 (n=432)
  Positive  ->  P: 0.8483 | R: 0.7402 | F1: 0.7906 (n=204)
Saved results to results/metrics/qlora.json and results/predictions/qlora.csv


### Direct Comparison: LoRA vs. QLoRA

By comparing `results/metrics/lora.json` and `results/metrics/qlora.json`:
1. **Memory:** QLoRA drastically lowers the VRAM requirement, allowing efficient fine-tuning on free cloud GPUs.
2. **Quality:** In accordance with Dettmers et al. (2023), the 4-bit NF4 quantized base model with LoRA adapters recovers virtually all the predictive accuracy of full-precision LoRA.
3. In the next notebook, we conduct a **rank ablation study** to see how changing $r$ affects performance and resources.